# Enunciado: "Teste ajustar o modelo de árvore de decisão e o modelo de Floresta aleatória e estime qual deles é mais apropriado usando um divisão de conjuntos de treinamento e teste de 70 e 30%."

In [1]:
# Instalar e importar bibliotecas
!pip install ucimlrepo --quiet

import pandas as pd
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns


In [5]:
abalone = fetch_ucirepo(id=1)

# Separar features e target
X = abalone.data.features.copy()
y = abalone.data.targets.copy()

# Converter variável categórica 'Sex' para numérica
le = LabelEncoder()
X['Sex'] = le.fit_transform(X['Sex'])

print("Dimensões de X:", X.shape)
print("Dimensões de y:", y.shape)

Dimensões de X: (4177, 8)
Dimensões de y: (4177, 1)


# Bloco 2 – Separar dois conjuntos de variáveis de decisão diferentes
# Enunciado: "Separe dois conjuntos de variáveis de decisão diferentes para um modelo regressor do número de anéis."

In [8]:
# Dividir os dados em treino e teste (70/30)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Criar dois subconjuntos de variáveis
X1_train = X_train.loc[:, ['Sex', 'Length', 'Height', 'Whole_weight']]
X2_train = X_train.loc[:, ['Diameter', 'Shucked_weight', 'Viscera_weight', 'Shell_weight']]

X1_test = X_test.loc[:, ['Sex', 'Length', 'Height', 'Whole_weight']]
X2_test = X_test.loc[:, ['Diameter', 'Shucked_weight', 'Viscera_weight', 'Shell_weight']]

print("Conjunto 1 (X1):", X1_train.columns.tolist())
print("Conjunto 2 (X2):", X2_train.columns.tolist())

Conjunto 1 (X1): ['Sex', 'Length', 'Height', 'Whole_weight']
Conjunto 2 (X2): ['Diameter', 'Shucked_weight', 'Viscera_weight', 'Shell_weight']


# Bloco 3 – Ajustar um modelo de árvore de decisão
# Enunciado: "Ajuste um modelo de árvore de decisão."

In [9]:
# Modelo para conjunto 1
tree1 = DecisionTreeRegressor(random_state=42)
tree1.fit(X1_train, y_train)
y_pred1 = tree1.predict(X1_test)

# Modelo para conjunto 2
tree2 = DecisionTreeRegressor(random_state=42)
tree2.fit(X2_train, y_train)
y_pred2 = tree2.predict(X2_test)

# Avaliar os dois
print("Conjunto 1 - Árvore de Decisão")
print("MSE:", mean_squared_error(y_test, y_pred1))
print("R²:", r2_score(y_test, y_pred1))
print("-"*40)
print("Conjunto 2 - Árvore de Decisão")
print("MSE:", mean_squared_error(y_test, y_pred2))
print("R²:", r2_score(y_test, y_pred2))

Conjunto 1 - Árvore de Decisão
MSE: 13.078149920255184
R²: -0.2879093979510785
----------------------------------------
Conjunto 2 - Árvore de Decisão
MSE: 9.647527910685806
R²: 0.0499312258285276


# Refaça o sorteio diversas vezes e produza diversos modelos

In [15]:
resultados = []

for seed in range(10):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=seed)

    # Corrigir formato do y (de DataFrame para vetor)
    y_train = y_train.values.ravel()
    y_test = y_test.values.ravel()

    X1_train = X_train.loc[:, ['Sex', 'Length', 'Height', 'Whole_weight']]
    X2_train = X_train.loc[:, ['Diameter', 'Shucked_weight', 'Viscera_weight', 'Shell_weight']]
    X1_test = X_test.loc[:, ['Sex', 'Length', 'Height', 'Whole_weight']]
    X2_test = X_test.loc[:, ['Diameter', 'Shucked_weight', 'Viscera_weight', 'Shell_weight']]

    # Árvore
    tree = DecisionTreeRegressor(random_state=seed)
    tree.fit(X1_train, y_train)
    y_pred_tree = tree.predict(X1_test)

    # Floresta
    forest = RandomForestRegressor(n_estimators=100, random_state=seed)
    forest.fit(X2_train, y_train)
    y_pred_forest = forest.predict(X2_test)

    resultados.append({
        "Seed": seed,
        "Modelo": "Árvore",
        "MSE": mean_squared_error(y_test, y_pred_tree),
        "R2": r2_score(y_test, y_pred_tree)
    })
    resultados.append({
        "Seed": seed,
        "Modelo": "Floresta",
        "MSE": mean_squared_error(y_test, y_pred_forest),
        "R2": r2_score(y_test, y_pred_forest)
    })

df_resultados = pd.DataFrame(resultados)
df_resultados.head()

,Seed,Modelo,MSE,R2
0,0,Árvore,12.362440,-0.172572
1,0,Floresta,5.392109,0.488561
2,1,Árvore,13.603668,-0.352882
3,1,Floresta,5.010455,0.501711
4,2,Árvore,13.690789,-0.201309


# Escolha um modelo e justifique sua escolha comentando possíveis problemas

Com base nos resultados obtidos, o modelo de Floresta Aleatória foi escolhido por apresentar menor erro (MSE) e maior coeficiente de determinação (R²), demonstrando melhor desempenho na predição do número de anéis do molusco Abalone.
A Árvore de Decisão, embora simples, mostrou R² negativo, indicando dificuldade em generalizar para novos dados.
A Floresta Aleatória supera esse problema ao combinar várias árvores independentes, reduzindo o risco de overfitting.
Como possíveis limitações, destacam-se o maior custo computacional e a menor interpretabilidade do modelo.